In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
from torch.nn.utils.rnn import pack_sequence, pack_padded_sequence, pad_sequence, unpack_sequence, pad_packed_sequence


import pandas as pd
import numpy as np
import ast

torch.manual_seed(42)


DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


In [2]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_layers, output_dim, bidirectional=False, device="cpu"):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.number_of_layers = number_of_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, number_of_layers, batch_first=True, device=device, bidirectional=bidirectional)
        self.fc = nn.Linear(hidden_dim, output_dim, device=device)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        final_hidden = hn[-1]
        # out, lens = pad_packed_sequence(out)
        # print(lens-1)
        # out = out[lens-1, ...]  # only take the output of the last element in the series which is the output of the lstm
        # out = out[-1, ...]
        out = self.sigmoid(final_hidden)
        out = self.fc(out)
        out = self.sigmoid(out)
        return out
    
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        self.fully_connected1 = nn.LazyLinear(1, device=DEVICE)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # out = x/14
        out = self.fully_connected1(x)
        out = self.sigmoid(out)
        return out

In [ ]:
# Initialize Datasets

class TelescopeSequenceModelingDataset(Dataset):
    def __init__(self, dataset_file):
        df = pd.read_csv(dataset_file)
        
        self.sequences_list = [torch.tensor(ast.literal_eval(string_sequence), dtype=torch.float32, device=DEVICE).view(-1, 1) for string_sequence in df["telescope_perplexity_per_token"]] 
        self.labels = torch.tensor(df["labels"].to_numpy(), dtype=torch.float32, device=DEVICE) 

    def __len__(self):
        return len(self.sequences_list)

    def __getitem__(self, idx):
        return self.sequences_list[idx], self.labels[idx]


class FullSequenceModelingDataset(Dataset):
    def __init__(self, dataset_file):
        df = pd.read_csv(dataset_file)

        self.sequences_list = []
        for telescope_perplexity_string_sequence, cross_perplexity_string_sequence, perplexity_string_sequence in zip(df["telescope_perplexity_per_token"], df["cross_perplexity_per_token"], df["perplexity_per_token"]):
            
            sequence = torch.tensor(
                list(zip(ast.literal_eval(telescope_perplexity_string_sequence)[0], ast.literal_eval(cross_perplexity_string_sequence)[0][:-1], ast.literal_eval(perplexity_string_sequence)[0])), 
                dtype=torch.float32, device=DEVICE
            )
            
            sequence = sequence / 14  # normalize for the neural network
                        
            self.sequences_list.append(sequence)        
        
        self.labels = torch.tensor(df["labels"].to_numpy(), dtype=torch.float32, device=DEVICE) 

    def __len__(self):
        return len(self.sequences_list)

    def __getitem__(self, idx):
        return self.sequences_list[idx], self.labels[idx]



class TelescopeAverageDataset(Dataset):
    def __init__(self, dataset_file):
        df = pd.read_csv(dataset_file)
        
        self.sequences_list = torch.tensor([np.average(ast.literal_eval(string_sequence)) for string_sequence in df["telescope_perplexity_per_token"]], dtype=torch.float32, device=DEVICE).view(-1, 1)
        self.labels = torch.tensor(df["labels"].to_numpy(), dtype=torch.float32, device=DEVICE) 

    def __len__(self):
        return len(self.sequences_list)

    def __getitem__(self, idx):
        return self.sequences_list[idx], self.labels[idx]


# pack batches properly so that the LSTM module can properly use it
def collate_fn(batch: list[tuple[torch.Tensor, float]]) -> torch.nn.utils.rnn.PackedSequence:
    packed_sequences = pack_sequence([sequence[0] for sequence in batch], enforce_sorted=False)
    labels = torch.tensor([sequence[1] for sequence in batch], device=DEVICE)
    return packed_sequences, labels
        
        
        
dataset_list = []
for dataset_name in ["hc3_plus_smollm_360M_dataset", "hc3_smollm_360M_dataset", "ai_human_smollm_360M_dataset", "detect_llm_text_smollm_360M_dataset", "esl_gpt4o_smollm_360M_dataset"]:
    dataset_list.append(TelescopeSequenceModelingDataset(f"sequence_modeling_datasets/{dataset_name}/full.csv"))
full_dataset = torch.utils.data.ConcatDataset(dataset_list)

train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)


     
# dataset_list = []
# for dataset_name in ["hc3_plus_smollm_360M_dataset", "hc3_smollm_360M_dataset", "ai_human_smollm_360M_dataset", "detect_llm_text_smollm_360M_dataset", "esl_gpt4o_smollm_360M_dataset"]:
#     dataset_list.append(TelescopeAverageDataset(f"sequence_modeling_datasets/{dataset_name}/full.csv"))
# full_dataset = torch.utils.data.ConcatDataset(dataset_list)

# train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
# train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)
# test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=True)





# full_dataset = FullSequenceModelingDataset("sequence_modeling_datasets/hc3_plus_smollm_360M_dataset/full.csv") 
# train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
# train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
# test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)



In [4]:
num_epochs = 30
    
model = LSTMModel(input_dim=1, hidden_dim=300, number_of_layers=3, output_dim=1, device=DEVICE, bidirectional=True)
# model = LogisticRegressionModel()

model.train()
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00003)

scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
# scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=num_epochs)


for epoch in range(num_epochs):
    for batch_index, (batch_data, batch_labels) in enumerate(train_dataloader):
        optimizer.zero_grad()

        outputs = model(batch_data)
        
        outputs = outputs.view(-1)
        
        loss = loss_function(outputs, batch_labels)
        
        loss.backward()
        optimizer.step()

        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/30], Loss: 0.6912
Epoch [1/30], Loss: 0.6815
Epoch [1/30], Loss: 0.7041


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.74 GiB. GPU 0 has a total capacity of 7.75 GiB of which 1.97 GiB is free. Including non-PyTorch memory, this process has 5.51 GiB memory in use. Of the allocated memory 1.78 GiB is allocated by PyTorch, and 3.57 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [57]:
correct = 0
total = 0

with torch.no_grad():
    for batch_index, (batch_data, batch_labels) in enumerate(test_dataloader):
        
        if batch_labels.shape[0] != 256: continue
        
        # print(batch_data)
        batch_output = model(batch_data)
        
        batch_output = batch_output.view(-1)
        batch_output = batch_output.cpu().numpy()
        
        batch_labels = batch_labels.cpu().numpy()        

        preds = batch_output > 0.5
        labels = batch_labels > 0.5
        correct += np.count_nonzero(preds == labels)
         
        total += 256
        
        print(correct/total)

0.9609375
0.95703125
0.9583333333333334
0.962890625
0.96484375
0.9674479166666666
0.9659598214285714
0.96826171875
0.9670138888888888
0.965234375
0.9659090909090909
0.9674479166666666
0.9678485576923077
0.9679129464285714
0.9671875
0.968505859375
0.96875
0.9678819444444444
0.9681332236842105
0.968359375
0.9681919642857143
0.9676846590909091
0.9684103260869565
0.96826171875
0.9684375
0.96875
0.9691840277777778
0.9698660714285714
0.9705010775862069
0.9705729166666667
0.9706401209677419
0.9708251953125
0.9707623106060606
0.9705882352941176
0.9703125


In [ ]:
torch.save(model, 'models/smollm_360M_lstm_extra_features_all_datasets.pt')


In [ ]:
# model = torch.load("models/hc3_plus_smollm_360M_model.pt")
model = torch.load("models/detect_llm_text_hc3_plus_smollm_360M_lstm.pt")


correct = 0
total = 0

with torch.no_grad():
    for batch_index, (batch_data, batch_labels) in enumerate(test_dataloader):
        
        print(batch_labels.shape[0])
        if batch_labels.shape[0] != 128: continue
        
        # print(batch_data)
        batch_output = model(batch_data)
        
        batch_output = batch_output.view(-1)
        batch_output = batch_output.cpu().numpy()
        
        batch_labels = batch_labels.cpu().numpy()        

        preds = batch_output > 0.5
        labels = batch_labels > 0.5
        correct += np.count_nonzero(preds == labels)
         
        total += 128
        
        print(correct/total)

/tmp/ipykernel_5982/2113162760.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("detectllmtext_hc3_plus_smollm_360M_lstm.pt")


128
0.984375
128
0.98828125
128
0.9869791666666666
128
0.98828125
128
0.9859375
128
0.98828125
128
0.9899553571428571
128
0.98828125
128
0.9869791666666666
128
0.98515625
128
0.9865056818181818
128
0.9869791666666666
128
0.9879807692307693
128
0.9888392857142857
128
0.9895833333333334
128
0.98974609375
128
0.9898897058823529
128
0.9900173611111112
128
0.9905427631578947
128
0.990234375
128
0.9899553571428571
128
0.9904119318181818
128
0.9898097826086957
128
0.9889322916666666
128
0.9890625
128
0.9894831730769231
128
0.9898726851851852
128
0.9899553571428571
128
0.9900323275862069
128
0.98984375
128
0.9896673387096774
32
